# 02 — Load online models and fine-tune

Run one model at a time so that the GPU is not overwhelmed. The notebook imports architecture code and checkpoints from online sources automatically on first use, then caches them.

## Step 1 — Project path and GPU

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd()
if not (ROOT / 'app').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'app').is_dir():
    raise FileNotFoundError('Open the complete iMAGE_Notebook_Edition folder in VS Code first.')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project:', ROOT)


In [ ]:
import torch
from app.experiments import dataset_counts, setup_device
device=setup_device()
print('PyTorch:',torch.__version__,'Device:',device)
print('CUDA available:',torch.cuda.is_available())
print(dataset_counts())


## Step 2 — Choose a model (repeat the notebook with each name)

DnCNN is the lightest starting point. Restormer/NAFNet require more GPU memory. The selected models use different published noise pretraining: NAFNet SIDD is real noise, while the default Restormer and DnCNN checkpoints are Gaussian noise. Fine-tune all on the **same** Gaussian noise dataset before drawing conclusions.

In [ ]:
MODEL='dncnn'  # 'dncnn', 'restormer', 'nafnet'
EPOCHS=5
BATCH_SIZE=2 if MODEL=='dncnn' else 1
PATCH=128
SIGMA=25
LEARNING_RATE=1e-5
VALIDATION_LIMIT=500  # use all 500 for the final tracked run
MAX_TRAIN_STEPS=0    # 0 = every training image; use 5 only for a smoke test
SEED=42
print('Selected:',MODEL)


## Step 3 — Automatically fetch model code and pretrained weights (first run needs internet)

No manual `git clone` or `.pth` download is needed. Model code is cached under `cache/upstream/`; weights under `cache/pretrained/`. The NAFNet checkpoint is mirrored on Hugging Face and verified against a known digest. Treat downloaded model code as third-party software.

In [ ]:
from app.online import architecture_path, pretrained_path
if MODEL in ('restormer','nafnet'):
    print('Architecture:',architecture_path(MODEL))
print('Pretrained weights:',pretrained_path(MODEL))


## Step 4 — Fine-tune a model and save best validation checkpoint

Do not run the full training loop until your GPU, dataset, and checkpoint have been confirmed. On a quick smoke test set `EPOCHS=1`, `MAX_TRAIN_STEPS=5`, and `VALIDATION_LIMIT=10`; then reset for full training.

In [ ]:
from app.experiments import fine_tune
RUN_TRAINING=False  # Change to True when ready
if RUN_TRAINING:
    history=fine_tune(MODEL,epochs=EPOCHS,batch_size=BATCH_SIZE,patch=PATCH,
                      sigma=SIGMA,lr=LEARNING_RATE,validation_limit=VALIDATION_LIMIT,
                      max_train_steps=MAX_TRAIN_STEPS,seed=SEED,
                      checkpoint=pretrained_path(MODEL))
else:
    print('Training not started. Change RUN_TRAINING=True after reading settings.')


## Step 5 — Plot training and validation curves; save the graph

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
log=ROOT/'results'/f'{MODEL}_training.csv'
if log.exists():
    df=pd.read_csv(log);display(df)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].plot(df.epoch,df.train_l1,marker='o');axes[0].set(xlabel='Epoch',ylabel='Training L1 loss',title='Training loss')
    axes[1].plot(df.epoch,df.validation_psnr_db,marker='o');axes[1].set(xlabel='Epoch',ylabel='Validation PSNR (dB)',title='Validation accuracy')
    fig.tight_layout();fig.savefig(ROOT/'results'/f'{MODEL}_training_curves.png',dpi=150);plt.show()
else:
    print('No training log yet. Run Step 4 first.')


After all three models have saved `weights/{model}_finetuned.pth`, proceed to notebook 03. Never evaluate on the test set while choosing hyperparameters.